# Step 5b: Add duplicate CRE sequences to count table

Some oCRE sequences appear as duplicates in the reference (same sequence, different tile IDs).
These duplicates weren't recovered in sequencing because they are identical, but their counts
should be attributed to all duplicate CRE IDs.

**Input:**
- Count table without zeroes from Step 5
- Duplicate CRE reference file

**Output:**
- Extended count table with duplicate tile counts added

In [ ]:
import pandas as pd
import glob
import ast

In [ ]:
# Load the count table from Step 5
count_table_path = 'output/PYS2_oCREv2_CRE_traj_MPRA_count_table_wo_zeroes.txt.gz'
df = pd.read_csv(count_table_path, sep='\t', low_memory=False)
df.head()

In [ ]:
df['class'].unique()

In [ ]:
# Read duplicate reference and add missing duplicate CRE counts
dup_ref_path = 'data/dictionaries/all_oCRE_cactusv2_reoriented_marked_dups.txt'

missing_dup_count = pd.DataFrame()
still_did_not_sequence = []

with open(dup_ref_path, 'r') as filehandle:
    for line in filehandle:
        dup_read = line.strip().split('\t')
        if dup_read[0] == 'tile_name':
            pass
        else:
            literal_dup = ast.literal_eval(dup_read[1])
            dup_read = [dup_read[0]] + literal_dup
            temp_df = df[df['CRE_id'].isin(dup_read)]
            if temp_df.shape[0] == 0:
                print('did not sequence')
                print(dup_read)
                still_did_not_sequence += dup_read
            else:
                uniq_cre_id = temp_df['CRE_id'].unique()
                if len(uniq_cre_id) == len(dup_read):
                    print('yes')
                else:
                    print(uniq_cre_id)
                    missing = [x for x in dup_read if x not in uniq_cre_id]
                    print(missing)
                    for x in missing:
                        new_df = temp_df.copy()
                        new_df['CRE_id'] = x
                        new_df['quantification_id'] = uniq_cre_id[0]
                        missing_dup_count = pd.concat([missing_dup_count, new_df])

In [ ]:
len(still_did_not_sequence)

In [ ]:
still_did_not_sequence

In [ ]:
# Write duplicate ID mapping
dup_id = missing_dup_count[['CRE_id', 'quantification_id']].drop_duplicates()
dup_id.to_csv('output/oCRE_duplicate_id_collapsed.txt', sep='\t', index=False)

In [ ]:
missing_dup_count = missing_dup_count.drop(['quantification_id'], axis=1)
missing_dup_count['is_duplicate'] = True
missing_dup_count.head()

In [ ]:
df_w_dups = pd.concat([df, missing_dup_count])

In [ ]:
print(f'Original rows: {df.shape[0]}')
print(f'After adding duplicates: {df_w_dups.shape[0]}')

In [ ]:
df_w_dups.to_csv('output/PYS2_oCREv2_CRE_traj_MPRA_count_table_wo_zeroes_w_dups.txt.gz',
                 sep='\t', compression='gzip', na_rep='NULL', index=False)